# Lesson 02 - Microsoft Agent Framework 탐색하기

**Microsoft Agent Framework (MAF)**는 AI 에이전트를 구축하기 위한 통합 프레임워크입니다. 네 가지 핵심 빌딩 블록으로 구성된 깔끔하고 조합 가능한 아키텍처를 제공합니다:

- **Client** – AI 모델 엔드포인트에 연결하고 통신을 처리합니다
- **Agent** – 클라이언트를 명령 및 도구 정의와 함께 래핑합니다
- **Tools** – 모델이 호출할 수 있는 사용자 정의 함수로 에이전트 기능을 확장합니다
- **Session** – 다중 턴 상호작용을 위한 대화 기록을 유지합니다

이번 강의에서는 이러한 개념을 사용하여 목적지 가용성을 확인하는 **여행 예약 에이전트**를 만들어 보겠습니다.


## 설정


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 에이전트 프레임워크 아키텍처 이해하기

Microsoft 에이전트 프레임워크는 계층화된 아키텍처를 따릅니다:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **클라이언트** – `AzureAIProjectAgentProvider`가 Azure OpenAI 배포에 연결합니다. 인증, 요청 형식 지정, 응답 구문 분석을 처리합니다.
2. **에이전트** – 클라이언트에서 `provider.create_agent()`를 통해 생성되며, 모델 액세스와 지침(시스템 프롬프트) 및 도구를 결합합니다.
3. **도구** – 에이전트가 호출하여 작업을 수행하거나 데이터를 가져오는 `@tool`로 장식된 Python 함수입니다.
4. **세션** – 대화 기록을 저장하여 다중 턴 대화가 가능하도록 하는 `agent.create_session()`로 생성된 `AgentSession` 객체입니다. 에이전트가 이전 콘텍스트를 기억합니다.

각 계층을 단계별로 구축해 보겠습니다.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool 데코레이터로 도구 추가하기

도구는 에이전트가 텍스트 생성 외의 작업을 수행할 수 있게 합니다. `@tool` 데코레이터는 일반 파이썬 함수를 에이전트가 호출할 수 있는 형태로 변환합니다.

핵심 내용:
- `Annotated[type, "설명"]`을 사용하여 모델이 각 매개변수를 이해하도록 합니다.
- docstring이 모델이 보는 도구 설명이 됩니다.
- `approval_mode="never_require"`는 사용자 확인 없이 도구가 자동으로 실행됨을 의미합니다.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 도구를 사용하여 에이전트 생성하기

이제 클라이언트, 지침, 도구를 결합하여 에이전트를 만듭니다. `instructions`는 시스템 프롬프트 역할을 하며 에이전트의 페르소나와 행동을 정의합니다.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 세션을 통한 다중 대화

`AgentSession` (`agent.create_session()`을 통해 생성됨)는 대화 내 모든 메시지를 추적합니다. 동일한 세션을 각 `agent.run()` 호출에 전달하면 에이전트가 전체 대화 기록에 접근할 수 있고 이전 메시지를 참고할 수 있습니다.

각 턴마다 에이전트가 가용성 확인 도구를 호출할 수 있도록 `tools=[check_destination_availability]`를 전달합니다.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 요약

이 수업에서는 Microsoft Agent Framework의 네 가지 핵심 요소를 살펴보았습니다:

| 개념 | 배운 내용 |
|---------|------------------|
| **클라이언트** | `AzureAIProjectAgentProvider`는 자격 증명 기반 인증으로 Azure OpenAI에 연결합니다 |
| **에이전트** | `provider.create_agent()`는 모델 연결을 지침 및 이름과 함께 묶습니다 |
| **도구** | `@tool` 데코레이터는 에이전트가 호출할 수 있는 Python 함수를 노출합니다 |
| **세션** | `agent.create_session()`은 여러 턴에 걸친 대화 기록을 유지합니다 |

이들 구성 요소는 자연스러운 대화를 나누고 외부 함수를 호출하며 문맥을 유지할 수 있는 에이전트를 만드는 데 함께 작용합니다 — 이는 이후 수업에서 더 발전된 에이전트 패턴의 기반입니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 저희는 정확성을 위해 노력하고 있으나, 자동 번역에는 오류나 부정확한 내용이 포함될 수 있음을 유의해 주시기 바랍니다. 원본 문서가 권위 있는 출처임을 참고하시기 바랍니다. 중요한 정보의 경우에는 전문적인 인간 번역을 권장합니다. 본 번역의 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
